# Search Service SDK Playground

This notebook is a hands-on playground for the SDK in this repository.

It gives you two ways to explore the harness:

1. A zero-setup in-memory demo with a deterministic model provider.
2. A CSV-driven path you can point at a real business dataset, including an ABR-style extract.

Recommended setup:

- Run `uv sync --dev` from the repository root.
- Use the repository virtual environment as the notebook kernel.
- Run cells top to bottom the first time.

The notebook is intentionally transparent: the index config, model provider, and result rendering helpers are all visible so you can tweak them in-place.

In [18]:
from __future__ import annotations

import csv
from pathlib import Path
from pprint import pprint
from typing import Any

from pydantic import BaseModel

from search_service import (
    ClassificationResult,
    ExtractionResult,
    IndexConfig,
    InMemoryAdapter,
    InteractionMode,
    QueryAnalyzer,
    SearchClient,
    SearchPolicy,
)
from search_service.schemas.config import ConfidenceThresholds
from search_service.schemas.enums import AmbiguityLevel


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
PROJECT_ROOT


PosixPath('/Users/williamryan/PROJECTS/search-service-take-1')

In [8]:
class DemoBusinessRecord(BaseModel):
    id: str
    name: str
    country: str
    state: str
    postcode: str
    status: str
    entity_type: str
    industry: str


DEMO_BUSINESSES: list[dict[str, str]] = [
    {
        "id": "1",
        "name": "Telstra Corporation NSW",
        "country": "AU",
        "state": "NSW",
        "postcode": "2000",
        "status": "active",
        "entity_type": "company",
        "industry": "telecommunications",
    },
    {
        "id": "2",
        "name": "Telstra Corporation VIC",
        "country": "AU",
        "state": "VIC",
        "postcode": "3000",
        "status": "active",
        "entity_type": "company",
        "industry": "telecommunications",
    },
    {
        "id": "3",
        "name": "Telstra Global",
        "country": "US",
        "state": "CA",
        "postcode": "94016",
        "status": "active",
        "entity_type": "company",
        "industry": "telecommunications",
    },
    {
        "id": "4",
        "name": "Optus Networks",
        "country": "AU",
        "state": "NSW",
        "postcode": "2000",
        "status": "active",
        "entity_type": "company",
        "industry": "telecommunications",
    },
    {
        "id": "5",
        "name": "Acme Civil Pty Ltd",
        "country": "AU",
        "state": "QLD",
        "postcode": "4000",
        "status": "inactive",
        "entity_type": "company",
        "industry": "construction",
    },
]


class NotebookDemoProvider:
    @property
    def model_name(self) -> str:
        return "demo/notebook-provider"

    def classify_query(
        self,
        query: str,
        expected_query_types: list[str],
        *,
        entity_types: list[str] | None = None,
        example_queries: list[str] | None = None,
    ) -> ClassificationResult:
        return ClassificationResult(query_type="entity_lookup", confidence=0.9)

    def extract_entities(
        self,
        query: str,
        *,
        entity_types: list[str] | None = None,
        filterable_fields: list[str] | None = None,
        canonical_filters: dict[str, list[str]] | None = None,
    ) -> ExtractionResult:
        q = query.lower().strip()
        canonical_filters = canonical_filters or {}
        filters: dict[str, Any] = {}

        for field_name in ("state", "country", "status", "entity_type"):
            for value in canonical_filters.get(field_name, []):
                if str(value).lower() in q:
                    filters[field_name] = value
                    break

        primary_subject = query.strip() or "business record"
        if "telstra" in q:
            primary_subject = "Telstra"
        elif "optus" in q:
            primary_subject = "Optus"
        elif "acme" in q:
            primary_subject = "Acme"

        if q == "telstra" or q.startswith("ambiguous "):
            cleaned = query.replace("ambiguous", "", 1).strip() or primary_subject
            return ExtractionResult(
                ambiguity=AmbiguityLevel.high,
                primary_subject=cleaned,
                target_resource_type="company",
                missing_fields=["state"],
            )

        return ExtractionResult(
            ambiguity=AmbiguityLevel.none,
            primary_subject=primary_subject,
            filters=filters,
            target_resource_type="company",
        )


def build_demo_index(*, interaction_mode: InteractionMode = InteractionMode.hitl):
    client = SearchClient()
    config = IndexConfig(
        name="demo_businesses",
        document_schema=DemoBusinessRecord,
        adapter=InMemoryAdapter(
            documents=list(DEMO_BUSINESSES),
            searchable_fields=["name", "industry"],
        ),
        searchable_fields=["name", "industry"],
        filterable_fields=["country", "state", "status", "entity_type", "industry"],
        display_fields=["name", "country", "state", "status", "entity_type", "industry"],
        id_field="id",
        entity_types=["company"],
        expected_query_types=["entity_lookup", "filter_search"],
        default_interaction_mode=interaction_mode,
        policy=SearchPolicy(
            max_iterations=3,
            max_branches=2,
            canonical_filters={
                "country": ["AU", "US"],
                "state": ["NSW", "VIC", "QLD", "CA"],
                "status": ["active", "inactive"],
                "entity_type": ["company"],
            },
            example_queries=["Telstra", "Telstra NSW active"],
            confidence_thresholds=ConfidenceThresholds(stop=0.72, escalate=0.28),
        ),
    )
    index = client.indexes.create(config, analyzer=QueryAnalyzer(NotebookDemoProvider()))
    return client, index


In [9]:
def compact_result_item(item) -> dict[str, Any]:
    return {
        "id": item.id,
        "title": item.title,
        "metadata": item.metadata,
    }


def show_result(result) -> None:
    print(f"status: {result.status.value}")
    print(f"original_query: {result.original_query}")
    print(f"interaction_mode: {result.interaction_mode.value}")
    if result.message:
        print(f"message: {result.message}")
    print(f"trace_id: {result.trace_id}")

    if result.query_analysis is not None:
        print("\nquery_analysis:")
        pprint(result.query_analysis.model_dump())

    if result.follow_up is not None:
        print("\nfollow_up:")
        pprint(result.follow_up.model_dump())

    print("\nresults:")
    if not result.results:
        print("  <none>")
    for item in result.results:
        pprint(compact_result_item(item))

    print("\nbranches:")
    if not result.branches:
        print("  <none>")
    for branch in result.branches:
        pprint(
            {
                "kind": branch.kind.value,
                "query": branch.query,
                "filters": branch.filters,
                "total_backend_hits": branch.total_backend_hits,
            }
        )


def show_trace(client: SearchClient, trace_id: str) -> None:
    trace = client.tracer.get(trace_id)
    if trace is None:
        print(f"No trace found for {trace_id}")
        return

    print(f"trace_id: {trace.trace_id}")
    print(f"final_status: {trace.final_status.value if trace.final_status else None}")
    print(f"iterations_used: {trace.iterations_used}")
    print(f"branches_used: {trace.branches_used}")
    print("steps:")
    for step in trace.steps:
        print(f"- {step.step_type.value}: {step.payload}")


## Zero-setup playground

This section uses a deterministic demo provider, so you can experience the harness without any external search service or model credentials.

The `ambiguous <name>` pattern intentionally forces a HITL follow-up. Queries like `Telstra NSW active` trigger the AITL path with extracted filters.

In [10]:
demo_hitl_client, demo_hitl_index = build_demo_index(interaction_mode=InteractionMode.hitl)
demo_aitl_client, demo_aitl_index = build_demo_index(interaction_mode=InteractionMode.aitl)

print("Direct search example")
direct_result = demo_hitl_index.search("Optus")
show_result(direct_result)

print("\nHITL example")
needs_input_result = demo_hitl_index.search("ambiguous Telstra")
show_result(needs_input_result)

if needs_input_result.follow_up is not None:
    continued_result = demo_hitl_index.continue_search(
        needs_input_result.trace_id,
        {"state": "NSW"},
    )
    print("\nAfter continue_search(state=NSW)")
    show_result(continued_result)

print("\nAITL example")
aitl_result = demo_aitl_index.search("Telstra NSW active")
show_result(aitl_result)
print("\nTrace for the AITL example")
show_trace(demo_aitl_client, aitl_result.trace_id)


Direct search example
status: completed
original_query: Optus
interaction_mode: hitl
message: Search completed with 1 result(s) (1 iteration(s), 2 branch(es))
trace_id: 98c549e5be71444981db9f0399d54528

query_analysis:
{'ambiguity': <AmbiguityLevel.none: 'none'>,
 'extracted_entities': [],
 'filters': {'country': 'US'},
 'missing_fields': [],
 'possible_resource_types': [],
 'primary_subject': 'Optus',
 'query_type': 'entity_lookup',
 'raw_query': 'Optus',
 'target_resource_type': 'company'}

results:
{'id': '4',
 'metadata': {'country': 'AU',
              'entity_type': 'company',
              'industry': 'telecommunications',
              'name': 'Optus Networks',
              'state': 'NSW',
              'status': 'active'},
 'title': 'Optus Networks'}

branches:
{'filters': {},
 'kind': 'original_query',
 'query': 'Optus',
 'total_backend_hits': 1}
{'filters': {'country': 'US'},
 'kind': 'filter_augmented',
 'query': 'Optus',
 'total_backend_hits': 0}

HITL example
status: nee

## Load your own CSV or delimited extract

This path is designed for what you described: grab a real business dataset, normalize it, and stand up an in-memory index quickly.

**Large files:** keep `CSV_LIMIT` set so you only load a sample into RAM. For multi-gigabyte ABR merges, prefer the Typesense section below instead of loading everything in-memory.

Notes:

- The loader will try to detect common delimiters, including commas and tabs.
- That means it can often handle both CSVs and ABR-style tab-delimited exports.
- The field mapping is suggestion-based, so you can override anything it guesses incorrectly.

In [11]:
class CsvBusinessRecord(BaseModel):
    id: str
    name: str
    country: str | None = None
    state: str | None = None
    postcode: str | None = None
    status: str | None = None
    entity_type: str | None = None
    industry: str | None = None


HEADER_ALIASES: dict[str, list[str]] = {
    "id": ["id", "abn", "abn_number", "business_id"],
    "name": [
        "name",
        "entity_name",
        "entityname",
        "legal_name",
        "business_name",
        "organisation_name",
        "main_name",
    ],
    "country": ["country", "country_code"],
    "state": [
        "state",
        "state_code",
        "main_business_physical_address_state_code",
        "address_state",
    ],
    "postcode": [
        "postcode",
        "post_code",
        "zip",
        "main_business_physical_address_postcode",
    ],
    "status": ["status", "abn_status", "entity_status"],
    "entity_type": ["entity_type", "entity_type_name", "type"],
    "industry": ["industry", "industry_name", "anzsic_description", "anzsic_code"],
}


def normalize_header(value: str) -> str:
    return "".join(ch.lower() for ch in value if ch.isalnum() or ch == "_")


def sniff_dialect(csv_path: Path) -> csv.Dialect:
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        sample = handle.read(4096)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",\t|;")
    except csv.Error:
        return csv.get_dialect("excel")


def read_csv_headers(csv_path: Path) -> list[str]:
    dialect = sniff_dialect(csv_path)
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, dialect=dialect)
        return list(reader.fieldnames or [])


def suggest_field_mapping(headers: list[str]) -> dict[str, str]:
    normalized_to_header = {normalize_header(header): header for header in headers}
    mapping: dict[str, str] = {}
    for target_field, aliases in HEADER_ALIASES.items():
        for alias in aliases:
            header = normalized_to_header.get(normalize_header(alias))
            if header is not None:
                mapping[target_field] = header
                break
    return mapping


def clean_value(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).strip()
    return text or None


def row_value(row: dict[str, Any], mapping: dict[str, str], field_name: str) -> str | None:
    column_name = mapping.get(field_name)
    if column_name is None:
        return None
    return clean_value(row.get(column_name))


def normalize_csv_row(
    row: dict[str, Any],
    mapping: dict[str, str],
    *,
    row_number: int,
) -> dict[str, str] | None:
    name = row_value(row, mapping, "name")
    if name is None:
        return None

    identifier = row_value(row, mapping, "id") or f"row-{row_number}"
    return {
        "id": identifier,
        "name": name,
        "country": row_value(row, mapping, "country") or "AU",
        "state": row_value(row, mapping, "state") or "",
        "postcode": row_value(row, mapping, "postcode") or "",
        "status": row_value(row, mapping, "status") or "unknown",
        "entity_type": row_value(row, mapping, "entity_type") or "unknown",
        "industry": row_value(row, mapping, "industry") or "",
    }


def load_csv_documents(
    csv_path: Path,
    mapping: dict[str, str],
    *,
    limit: int | None = None,
) -> list[dict[str, str]]:
    dialect = sniff_dialect(csv_path)
    documents: list[dict[str, str]] = []
    with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle, dialect=dialect)
        for row_number, row in enumerate(reader, start=1):
            normalized = normalize_csv_row(row, mapping, row_number=row_number)
            if normalized is None:
                continue
            documents.append(normalized)
            if limit is not None and len(documents) >= limit:
                break
    return documents


def collect_canonical_filters(
    documents: list[dict[str, str]],
    fields: list[str],
    *,
    max_values: int = 25,
) -> dict[str, list[str]]:
    canonical: dict[str, list[str]] = {}
    for field_name in fields:
        values = sorted({doc[field_name] for doc in documents if doc.get(field_name)})
        if values:
            canonical[field_name] = values[:max_values]
    return canonical


def build_csv_index(
    documents: list[dict[str, str]],
    *,
    interaction_mode: InteractionMode = InteractionMode.hitl,
    with_demo_analyzer: bool = True,
    index_name: str = "abr_entities",
):
    if not documents:
        raise ValueError("No documents were loaded from the file")

    client = SearchClient()
    canonical_filters = collect_canonical_filters(
        documents,
        ["country", "state", "status", "entity_type"],
    )
    config = IndexConfig(
        name=index_name,
        document_schema=CsvBusinessRecord,
        adapter=InMemoryAdapter(documents=documents, searchable_fields=["name", "industry"]),
        searchable_fields=["name", "industry"],
        filterable_fields=["country", "state", "status", "entity_type"],
        display_fields=["name", "country", "state", "postcode", "status", "entity_type", "industry"],
        id_field="id",
        entity_types=["company"],
        expected_query_types=["entity_lookup", "filter_search"],
        default_interaction_mode=interaction_mode,
        policy=SearchPolicy(
            max_iterations=3,
            max_branches=2,
            canonical_filters=canonical_filters,
            example_queries=["Telstra", "active companies in NSW"],
            confidence_thresholds=ConfidenceThresholds(stop=0.72, escalate=0.28),
        ),
    )
    analyzer = QueryAnalyzer(NotebookDemoProvider()) if with_demo_analyzer else None
    index = client.indexes.create(config, analyzer=analyzer)
    return client, index


In [12]:
CSV_PATH = PROJECT_ROOT / "data" / "abr_merged.csv"
CSV_LIMIT = 5000
CSV_QUERY = "Telstra"
CSV_FOLLOW_UP_INPUT = {"state": "NSW"}

if CSV_PATH.exists():
    csv_headers = read_csv_headers(CSV_PATH)
    FIELD_MAPPING = suggest_field_mapping(csv_headers)
    print(f"Detected file: {CSV_PATH}")
    print("\nHeaders:")
    pprint(csv_headers[:25])
    print("\nSuggested field mapping:")
    pprint(FIELD_MAPPING)
else:
    FIELD_MAPPING = {}
    print("Set CSV_PATH to a real dataset file, then re-run this cell.")
    print("ABR-style tab-delimited extracts are often fine too because the loader sniffs delimiters.")


Detected file: /Users/williamryan/PROJECTS/search-service-take-1/data/abr_merged.csv

Headers:
['source_file',
 'record_last_updated_date',
 'replaced',
 'abn',
 'abn_status',
 'abn_status_from_date',
 'entity_type_ind',
 'entity_type_text',
 'entity_name',
 'entity_name_type',
 'main_name',
 'legal_full_name',
 'legal_name_title',
 'legal_given_names',
 'legal_family_name',
 'legal_name_suffix',
 'state',
 'postcode',
 'asic_number',
 'asic_number_type',
 'gst_status',
 'gst_status_from_date',
 'dgr_status',
 'dgr_status_from_date',
 'trading_names']

Suggested field mapping:
{'id': 'abn',
 'name': 'entity_name',
 'postcode': 'postcode',
 'state': 'state',
 'status': 'abn_status'}


In [13]:
if not CSV_PATH.exists():
    print("Update CSV_PATH first, then run this cell.")
else:
    csv_documents = load_csv_documents(CSV_PATH, FIELD_MAPPING, limit=CSV_LIMIT)
    print(f"Loaded {len(csv_documents)} normalized documents")
    print("\nFirst 3 normalized rows:")
    pprint(csv_documents[:3])

    csv_client, csv_index = build_csv_index(
        csv_documents,
        interaction_mode=InteractionMode.hitl,
        with_demo_analyzer=True,
    )

    print(f"\nRunning query: {CSV_QUERY!r}")
    csv_result = csv_index.search(CSV_QUERY)
    show_result(csv_result)

    if csv_result.follow_up is not None:
        print(f"\nContinuing search with: {CSV_FOLLOW_UP_INPUT}")
        csv_continued = csv_index.continue_search(csv_result.trace_id, CSV_FOLLOW_UP_INPUT)
        show_result(csv_continued)

    print("\nTip: try queries like 'ambiguous Telstra', 'Telstra NSW active', or a real company name from your file.")


Loaded 5000 normalized documents

First 3 normalized rows:
[{'country': 'AU',
  'entity_type': 'unknown',
  'id': '11000000948',
  'industry': '',
  'name': 'QBE INSURANCE (INTERNATIONAL) LTD',
  'postcode': '2000',
  'state': 'NSW',
  'status': 'ACT'},
 {'country': 'AU',
  'entity_type': 'unknown',
  'id': '11000002568',
  'industry': '',
  'name': 'TOOHEYS PTY LIMITED',
  'postcode': '2141',
  'state': 'NSW',
  'status': 'CAN'},
 {'country': 'AU',
  'entity_type': 'unknown',
  'id': '11000003314',
  'industry': '',
  'name': 'NEWCASTLE GOLF CLUB LTD',
  'postcode': '2295',
  'state': 'NSW',
  'status': 'ACT'}]

Running query: 'Telstra'
status: needs_input
original_query: Telstra
interaction_mode: hitl
message: Additional information needed to refine search results
trace_id: f294cf8aa1e8461d9add91ddd5e37c26

query_analysis:
{'ambiguity': <AmbiguityLevel.high: 'high'>,
 'extracted_entities': [],
 'filters': {},
 'missing_fields': ['state'],
 'possible_resource_types': [],
 'primary_sub

## ABR dataset with Typesense

This section uses your real `data/abr_merged.csv` file and the `TypesenseAdapter` for a proper indexed backend.

**The ABR merge file can be multiple gigabytes.** Nothing here reads the whole file unless you explicitly opt in.

- **Metadata only:** the config cell prints size and headers from a single read of the first line (no full scan).
- **Row preview:** optional; bounded by `ABR_PREVIEW_MAX_SCAN_ROWS` so we never walk the entire CSV just to show two rows.
- **Import:** set `ABR_IMPORT_LIMIT` to a number (for example `25_000`) while you iterate. Full import requires `ABR_IMPORT_LIMIT = None` **and** `ABR_ALLOW_FULL_IMPORT = True`.
- **Reindex:** set `REINDEX_ABR_COLLECTION = True` only when you want to drop and reload the Typesense collection.
- **Later runs:** keep `REINDEX_ABR_COLLECTION = False` if the index already has data and you only want to query.

The helper streams the CSV in batches during import, creates the Typesense collection from `IndexConfig`, and upserts documents.

In [14]:
import sys
from pprint import pprint

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from examples.abr_typesense_helpers import (
    abr_csv_metadata,
    build_abr_typesense_index,
    ensure_typesense_container,
    get_collection_document_count,
    preview_abr_documents,
)

ABR_PATH = PROJECT_ROOT / "data" / "abr_merged.csv"
TYPESENSE_HOST = "localhost"
TYPESENSE_PORT = 8108
TYPESENSE_PROTOCOL = "http"
TYPESENSE_API_KEY = "search-service-dev"
TYPESENSE_COLLECTION = "abr_entities"
TYPESENSE_CONTAINER_NAME = "search-service-typesense"
TYPESENSE_DATA_DIR = PROJECT_ROOT / ".typesense-data"
TYPESENSE_IMAGE = "typesense/typesense:30.1"

START_TYPESENSE_CONTAINER = True
REINDEX_ABR_COLLECTION = False
ABR_IMPORT_LIMIT = 25_000
ABR_ALLOW_FULL_IMPORT = False
ABR_BATCH_SIZE = 5_000
ABR_LOAD_ROW_PREVIEW = True
ABR_PREVIEW_MAX_SCAN_ROWS = 500_000
ABR_RUN_QUICK_QUERIES = False
ABR_QUERY = "QBE NSW active"
ABR_FOLLOW_UP_INPUT = {"state": "NSW"}
ABR_QUICK_QUERIES = [
    "QBE NSW active",
    "Tooheys NSW cancelled",
    "private company in VIC",
    "sole trader in QLD",
    "gst registered business in 2000",
    "ambiguous Telstra",
]

print(f"ABR file exists: {ABR_PATH.exists()}")
print(f"ABR file: {ABR_PATH}")
if ABR_PATH.exists():
    meta = abr_csv_metadata(ABR_PATH)
    print(f"Size: {meta['size_gb']} GB ({meta['size_bytes']:,} bytes)")
    print("Headers (first line only):", meta["headers"][:12], "...")
if ABR_LOAD_ROW_PREVIEW and ABR_PATH.exists():
    print("\nPreview rows (bounded scan, not full file):")
    pprint(preview_abr_documents(ABR_PATH, limit=2, max_rows_to_scan=ABR_PREVIEW_MAX_SCAN_ROWS))
elif not ABR_LOAD_ROW_PREVIEW:
    print("\nSkipping row preview (ABR_LOAD_ROW_PREVIEW is False).")
print("\nStarter queries:")
pprint(ABR_QUICK_QUERIES)
if ABR_IMPORT_LIMIT is None and not ABR_ALLOW_FULL_IMPORT:
    raise ValueError("Set ABR_ALLOW_FULL_IMPORT = True to import the entire CSV, or set ABR_IMPORT_LIMIT to an integer.")


ABR file exists: True
ABR file: /Users/williamryan/PROJECTS/search-service-take-1/data/abr_merged.csv
Size: 4.181 GB (4,489,480,379 bytes)
Headers (first line only): ['source_file', 'record_last_updated_date', 'replaced', 'abn', 'abn_status', 'abn_status_from_date', 'entity_type_ind', 'entity_type_text', 'entity_name', 'entity_name_type', 'main_name', 'legal_full_name'] ...

Preview rows (bounded scan, not full file):
[{'abn': '11000000948',
  'abn_status': 'ACT',
  'all_other_entity_names': 'QBE INSURANCE (INTERNATIONAL) LIMITED',
  'business_names': '',
  'dgr_status': '',
  'entity_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
  'entity_name_type': 'MN',
  'entity_type_ind': 'PUB',
  'entity_type_text': 'Australian Public Company',
  'gst_status': 'ACT',
  'id': '11000000948',
  'legal_full_name': '',
  'main_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
  'other_names': '',
  'postcode': '2000',
  'record_last_updated_date': '20180216',
  'replaced': 'N',
  'source_file': 'public_split

In [15]:
if START_TYPESENSE_CONTAINER:
    ensure_typesense_container(
        container_name=TYPESENSE_CONTAINER_NAME,
        host_port=TYPESENSE_PORT,
        api_key=TYPESENSE_API_KEY,
        image=TYPESENSE_IMAGE,
        data_dir=TYPESENSE_DATA_DIR,
    )
    print("Typesense container is running.")
else:
    print("Skipping container startup. Make sure Typesense is already running.")

abr_client, abr_index, abr_ts_client, abr_import_summary = build_abr_typesense_index(
    csv_path=ABR_PATH,
    host=TYPESENSE_HOST,
    port=TYPESENSE_PORT,
    api_key=TYPESENSE_API_KEY,
    protocol=TYPESENSE_PROTOCOL,
    interaction_mode=InteractionMode.aitl,
    index_name=TYPESENSE_COLLECTION,
    ensure_collection=True,
    reindex=REINDEX_ABR_COLLECTION,
    import_limit=ABR_IMPORT_LIMIT,
    allow_full_import=ABR_ALLOW_FULL_IMPORT,
    batch_size=ABR_BATCH_SIZE,
    with_demo_analyzer=True,
)

print("Import summary:")
pprint(abr_import_summary)
print("\nCollection document count:", get_collection_document_count(abr_ts_client, TYPESENSE_COLLECTION))


Typesense container is running.
Import summary:
{'batch_size': 5000,
 'failed': 0,
 'imported': 25000,
 'limit': 25000,
 'sample_failures': []}

Collection document count: 25000


## ABR query recipes

These examples are tuned to the real ABR fields in `abr_merged.csv`.

Good starter patterns:

- `QBE NSW active`
- `Tooheys NSW cancelled`
- `private company in VIC`
- `sole trader in QLD`
- `gst registered business in 2000`
- `ambiguous Telstra`

What the demo analyzer currently understands:

- `active` / `cancelled` -> `abn_status`
- `gst registered` / `not gst registered` -> `gst_status`
- `dgr` / `deductible gift recipient` -> `dgr_status`
- `NSW`, `VIC`, `QLD`, etc. -> `state`
- 4-digit values like `2000` -> `postcode`
- `private company`, `public company`, `sole trader`, `partnership`, `smsf` -> `entity_type_text`

For exact power-user queries, you can always bypass the demo extraction and pass filters directly into `abr_index.search(..., filters={...})`.

The optional multi-query cell at the end is off by default (`ABR_RUN_QUICK_QUERIES = False`) so you do not accidentally run many orchestrated searches.

In [26]:
abr_result.model_dump()

{'status': <SearchStatus.completed: 'completed'>,
 'original_query': 'QBE NSW active',
 'interaction_mode': <InteractionMode.aitl: 'aitl'>,
 'query_analysis': {'raw_query': 'QBE NSW active',
  'query_type': 'entity_lookup',
  'primary_subject': 'QBE NSW active',
  'target_resource_type': 'company',
  'possible_resource_types': [],
  'filters': {'state': 'NSW', 'abn_status': 'ACT'},
  'ambiguity': <AmbiguityLevel.none: 'none'>,
  'missing_fields': [],
  'extracted_entities': []},
 'results': [{'id': '11000000948',
   'title': 'QBE INSURANCE (INTERNATIONAL) LTD',
   'snippet': None,
   'score': None,
   'source': 'abr_entities',
   'matched_fields': [],
   'metadata': {'abn': '11000000948',
    'entity_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
    'main_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
    'entity_type_text': 'Australian Public Company',
    'abn_status': 'ACT',
    'state': 'NSW',
    'postcode': '2000',
    'gst_status': 'ACT',
    'trading_names': 'QBE INSURANCE (INTERNAT

In [16]:
abr_result = abr_index.search(ABR_QUERY)
show_result(abr_result)

if abr_result.follow_up is not None:
    print(f"\nContinuing search with: {ABR_FOLLOW_UP_INPUT}")
    abr_continued = abr_index.continue_search(abr_result.trace_id, ABR_FOLLOW_UP_INPUT)
    show_result(abr_continued)

print("\nTrace for the ABR Typesense query")
show_trace(abr_client, abr_result.trace_id)

print("\nDirect filter-only examples you can try next:")
print("abr_index.search('Telstra', filters={'state': 'NSW', 'abn_status': 'ACT'})")
print("abr_index.search('QBE', filters={'entity_type_text': 'Australian Public Company'})")
print("abr_index.search('business', filters={'gst_status': 'ACT', 'postcode': '2000'})")
print("abr_index.search('trader', filters={'entity_type_text': 'Individual/Sole Trader', 'state': 'QLD'})")


status: completed
original_query: QBE NSW active
interaction_mode: aitl
message: Search completed with 2 result(s) (1 iteration(s), 2 branch(es))
trace_id: 322e1fa4849943b2b51d3c23b62664a5

query_analysis:
{'ambiguity': <AmbiguityLevel.none: 'none'>,
 'extracted_entities': [],
 'filters': {'abn_status': 'ACT', 'state': 'NSW'},
 'missing_fields': [],
 'possible_resource_types': [],
 'primary_subject': 'QBE NSW active',
 'query_type': 'entity_lookup',
 'raw_query': 'QBE NSW active',
 'target_resource_type': 'company'}

results:
{'id': '11000000948',
 'metadata': {'abn': '11000000948',
              'abn_status': 'ACT',
              'business_names': '',
              'entity_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
              'entity_type_text': 'Australian Public Company',
              'gst_status': 'ACT',
              'main_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
              'postcode': '2000',
              'state': 'NSW',
              'trading_names': 'QBE INSURANCE (

In [17]:
if not ABR_RUN_QUICK_QUERIES:
    print("Skipping quick-query sweep (set ABR_RUN_QUICK_QUERIES = True to run all starter queries).")
else:
    for quick_query in ABR_QUICK_QUERIES:
        print(f"\n=== {quick_query} ===")
        quick_result = abr_index.search(quick_query)
        print(f"status: {quick_result.status.value} | results: {len(quick_result.results)}")
        print([item.title for item in quick_result.results[:5]])
        if quick_result.follow_up is not None:
            print("follow_up required:", quick_result.follow_up.message)


Skipping quick-query sweep (set ABR_RUN_QUICK_QUERIES = True to run all starter queries).
